In [1]:
!pip uninstall -y protobuf tensorflow tensorflow-cpu tf-keras keras
!pip install -q protobuf==5.29.4
!pip install -q "transformers<5" datasets sentencepiece accelerate evaluate sacrebleu openpyxl

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Found existing installation: tf_keras 2.19.0
Uninstalling tf_keras-2.19.0:
  Successfully uninstalled tf_keras-2.19.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
tensorflow-hub 0.16.1 requires tf-keras>=2.14.1, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 M

In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import sacrebleu

In [34]:
TRAIN_FILE = "train_t5_clean.csv"
VALID_FILE = "valid_t5_clean.csv"
TEST_FILE  = "test_t5_clean.csv"

train_df = pd.read_csv(TRAIN_FILE)
valid_df = pd.read_csv(VALID_FILE)
test_df  = pd.read_csv(TEST_FILE)

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

train_df.head(10)

Train: (122, 2)
Valid: (13, 2)
Test : (13, 2)


,input_text,target_text
0,translate Arabic to gloss: انتبه,انتبه خطر
1,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل نص ساعة حبة
2,translate Arabic to gloss: حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة
3,translate Arabic to gloss: حبة 3 مرة قبل الاكل...,3 مرة اكل قبل ساعة حبة
4,translate Arabic to gloss: 6 شهر,6 شهر
5,translate Arabic to gloss: نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة
6,translate Arabic to gloss: لمدة ثلاثة يوم,استمرار ثلاثة يوم بس
7,translate Arabic to gloss: خده ورا الاكل,اكل بعد حبة معدة فاضية لا
8,translate Arabic to gloss: بعد الاكل,اكل خلص فورا دواء
9,translate Arabic to gloss: ممكن يعملك شوية اسه...,انتبه جسم تعب اسهال امساك دوخة


In [35]:
def clean_t5_df(df_):
    df_ = df_.dropna(subset=["input_text", "target_text"]).copy()
    df_["input_text"] = df_["input_text"].astype(str).str.strip()
    df_["target_text"] = df_["target_text"].astype(str).str.strip()
    df_ = df_[(df_["input_text"] != "") & (df_["target_text"] != "")].copy()
    df_ = df_.drop_duplicates(subset=["input_text", "target_text"]).reset_index(drop=True)
    return df_

train_df = clean_t5_df(train_df)
valid_df = clean_t5_df(valid_df)
test_df  = clean_t5_df(test_df)

print("After cleaning:")
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

After cleaning:
Train: (122, 2)
Valid: (13, 2)
Test : (13, 2)


In [36]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(valid_df),
    "test": Dataset.from_pandas(test_df),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 122
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 13
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 13
    })
})

In [37]:
MODEL_NAME = "UBC-NLP/AraT5v2-base-1024"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Tokenizer and model loaded.")

Tokenizer and model loaded.


In [38]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

In [39]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [40]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

Map:   0%|          | 0/13 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 122
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13
    })
})

In [41]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [42]:
import evaluate

bleu = evaluate.load("sacrebleu")

In [43]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

import numpy as np
import sacrebleu

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def sanitize_ids(arr, pad_token_id, vocab_size):
    arr = np.array(arr)

    # أي قيمة سالبة أو أكبر من آخر id صالح -> pad_token_id
    arr = np.where(arr < 0, pad_token_id, arr)
    arr = np.where(arr >= vocab_size, pad_token_id, arr)

    return arr

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # تنظيف القيم قبل فكّها
    preds = sanitize_ids(preds, tokenizer.pad_token_id, tokenizer.vocab_size)
    labels = sanitize_ids(labels, tokenizer.pad_token_id, tokenizer.vocab_size)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    bleu_score = sacrebleu.corpus_bleu(decoded_preds, decoded_labels).score

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]

    return {
        "bleu": round(float(bleu_score), 4),
        "gen_len": round(float(np.mean(prediction_lens)), 4),
    }

In [44]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./arat5_text2gloss",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=5e-5,
    num_train_epochs=10,
    weight_decay=0.01,

    predict_with_generate=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none"
)

In [45]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=None,
)

/tmp/ipykernel_3387/4008418194.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [46]:
print("pad_token_id:", tokenizer.pad_token_id)
print("vocab_size:", tokenizer.vocab_size)

sample = tokenized_dataset["train"][0]
print("max input id :", max(sample["input_ids"]))
print("max label id :", max([x for x in sample["labels"] if x != -100], default=-1))

pad_token_id: 0
vocab_size: 110000
max input id : 109590
max label id : 18797


In [47]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,33.645900,9.632418
2,10.879900,9.006188
3,8.972500,8.574604
4,8.522200,8.088944
5,7.961200,6.259936
6,6.793700,4.835260
7,5.593100,3.561639
8,4.866500,2.871164
9,4.307500,2.519088
10,4.009700,2.407021


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=310, training_loss=9.555215035715412, metrics={'train_runtime': 1336.4721, 'train_samples_per_second': 0.913, 'train_steps_per_second': 0.232, 'total_flos': 265036211159040.0, 'train_loss': 9.555215035715412, 'epoch': 10.0})

In [48]:
val_metrics = trainer.evaluate(tokenized_dataset["validation"])
val_metrics

{'eval_loss': 2.4070210456848145,
 'eval_runtime': 0.4151,
 'eval_samples_per_second': 31.321,
 'eval_steps_per_second': 9.637,
 'epoch': 10.0}

In [49]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])
test_metrics

{'eval_loss': 2.3846867084503174,
 'eval_runtime': 0.4357,
 'eval_samples_per_second': 29.837,
 'eval_steps_per_second': 9.181,
 'epoch': 10.0}

In [22]:
SAVE_DIR = "./arat5_text2gloss_best"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model to:", SAVE_DIR)

Saved model to: ./arat5_text2gloss_best


In [50]:
import torch
import pandas as pd
import sacrebleu

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

sample_test = test_df.copy()

inputs = tokenizer(
    sample_test["input_text"].tolist(),
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_INPUT_LENGTH
)

inputs = {k: v.to(device) for k, v in inputs.items()}

generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=MAX_TARGET_LENGTH,
    num_beams=4
)

preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

results_df = pd.DataFrame({
    "input_text": sample_test["input_text"].tolist(),
    "target_text": sample_test["target_text"].tolist(),
    "predicted_text": preds
})

results_df.head(20)


,input_text,target_text,predicted_text
0,translate Arabic to gloss: معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب,chem حبة
1,translate Arabic to gloss: قبل او بعد الاكل,اكل قبل اكل بعد عادي دواء خذ اي وقت,chem حبة
2,translate Arabic to gloss: عند الضرورة .,جسم تعب ضرورة بس دواء,chem حبة
3,translate Arabic to gloss: لازم تقطعيه,دواء وقف - فورا,chem حبة
4,translate Arabic to gloss: استمر ع البخاخ اسبو...,دواء استمرار ثلاثة اسبوع كامل دواء هدف راحة صح,chem حبة
5,translate Arabic to gloss: بلش فيه فورا,دواء بداية الان,chem حبة
6,"translate Arabic to gloss: دير بالك من الضوء ,...",دواء داخل كرتونة ضوء شمس لا - خطر,chem حبة
7,translate Arabic to gloss: ويفضل تبعد عن الشغل...,شرب بارد لا - ممنوع,chem حبة
8,translate Arabic to gloss: حصرا الصبح,صباح بكير حبة لازم,poetry حبة
9,translate Arabic to gloss: القطرة امنة,قطرة امن مشاكل - جسم - لا,chem حبة


In [51]:
predictions = results_df["predicted_text"].astype(str).str.strip().tolist()
references = [results_df["target_text"].astype(str).str.strip().tolist()]

bleu_score = sacrebleu.corpus_bleu(predictions, references).score
print("Test BLEU:", round(bleu_score, 4))

Test BLEU: 0.0


In [28]:
results_df["predicted_text"].value_counts().head(20)

,count
predicted_text,
estock تمام,10
chem تمام,2
estock تمام تمام,1


In [29]:
results_df["pred_len"] = results_df["predicted_text"].astype(str).apply(lambda x: len(x.split()))
results_df["target_len"] = results_df["target_text"].astype(str).apply(lambda x: len(x.split()))

print(results_df[["pred_len", "target_len"]].describe())

        pred_len  target_len
count  13.000000   13.000000
mean    2.076923    6.846154
std     0.277350    2.075498
min     2.000000    4.000000
25%     2.000000    5.000000
50%     2.000000    7.000000
75%     2.000000    9.000000
max     3.000000   10.000000


In [30]:
for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(f"\n===== {df_name.upper()} =====")

    for term in ["estock", "chem", "تمام"]:
        in_input = df_["input_text"].astype(str).str.contains(term, case=False, na=False).sum()
        in_target = df_["target_text"].astype(str).str.contains(term, case=False, na=False).sum()

        print(f"{term} -> input: {in_input}, target: {in_target}")

        if in_input > 0:
            print(f"\nSamples with '{term}' in input:")
            display(df_[df_["input_text"].astype(str).str.contains(term, case=False, na=False)].head(5))

        if in_target > 0:
            print(f"\nSamples with '{term}' in target:")
            display(df_[df_["target_text"].astype(str).str.contains(term, case=False, na=False)].head(5))


===== TRAIN =====
estock -> input: 0, target: 0
chem -> input: 0, target: 0
تمام -> input: 1, target: 73

Samples with 'تمام' in input:


,input_text,target_text
70,translate Arabic to gloss: استني ليروح الفوران...,كاسة ماء فوار انتظر غازة خلص اشرب



Samples with 'تمام' in target:


,input_text,target_text
0,translate Arabic to gloss: انتبه,انتبه خطر تمام
6,translate Arabic to gloss: لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام
7,translate Arabic to gloss: خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا
8,translate Arabic to gloss: بعد الاكل,اكل خلص فورا دواء تمام
9,translate Arabic to gloss: ممكن يعملك شوية اسه...,انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام



===== VALID =====
estock -> input: 0, target: 0
chem -> input: 0, target: 0
تمام -> input: 1, target: 7

Samples with 'تمام' in input:


,input_text,target_text
9,translate Arabic to gloss: استمر بالتنفس بعمق ...,نفس عميق استمرار دواء كلو خلص تمام



Samples with 'تمام' in target:


,input_text,target_text
0,translate Arabic to gloss: مع اكلة دسمة .,اكل دسم ثقيل لازم تمام بيض حليب زيت تمام دواء ...
2,translate Arabic to gloss: بخاخ مرتين باليوم,دواء انف بخ تمام يمين واحد يسار واحد تمام تكرا...
3,translate Arabic to gloss: لونه وريحته بس يتغي...,دواء لون غير تمام ريحة سيء تمام دواء انتزع تما...
4,translate Arabic to gloss: حبة بالنهار بعد الا...,كل يوم حبة اكل بعد تمام استمرار ثلاثة يوم بس
6,translate Arabic to gloss: حبة تحت اللسان : لا...,حبة تحت لسان بلع لا ممنوع تمام ذوبان وحدها لازم



===== TEST =====
estock -> input: 0, target: 0
chem -> input: 0, target: 0
تمام -> input: 0, target: 7

Samples with 'تمام' in target:


,input_text,target_text
0,translate Arabic to gloss: معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام
1,translate Arabic to gloss: قبل او بعد الاكل,اكل قبل اكل بعد تمام عادي دواء خذ اي وقت
3,translate Arabic to gloss: لازم تقطعيه,دواء تمام وقف - فورا
4,translate Arabic to gloss: استمر ع البخاخ اسبو...,دواء استمرار ثلاثة اسبوع كامل تمام دواء هدف را...
5,translate Arabic to gloss: بلش فيه فورا,دواء بداية الان تمام


In [31]:
import pandas as pd
import re

def remove_tamam(text: str) -> str:
    text = str(text)

    # حذف الكلمة "تمام" كتوكن مستقل
    text = re.sub(r"\bتمام\b", " ", text)

    # تنظيف المسافات والشرطات المكسورة
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

for df_ in [train_df, valid_df, test_df]:
    df_["target_text"] = df_["target_text"].astype(str).apply(remove_tamam)

print("Removed 'تمام' from target_text.")

Removed 'تمام' from target_text.


In [32]:
for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    count_tamam = df_["target_text"].astype(str).str.contains(r"\bتمام\b", regex=True, na=False).sum()
    print(df_name, "remaining 'تمام':", count_tamam)

train remaining 'تمام': 0
valid remaining 'تمام': 0
test remaining 'تمام': 0


In [33]:
train_df.to_csv("train_t5_clean.csv", index=False, encoding="utf-8-sig")
valid_df.to_csv("valid_t5_clean.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("test_t5_clean.csv", index=False, encoding="utf-8-sig")

print("Saved cleaned files.")

Saved cleaned files.


In [52]:
import re

for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    mask = df_["target_text"].astype(str).str.contains(r"[A-Za-z]", regex=True, na=False)
    print(f"\n===== {df_name.upper()} =====")
    print("Rows with Latin chars in target:", mask.sum())
    if mask.sum() > 0:
        display(df_[mask][["input_text", "target_text"]].head(30))


===== TRAIN =====
Rows with Latin chars in target: 0

===== VALID =====
Rows with Latin chars in target: 0

===== TEST =====
Rows with Latin chars in target: 0


In [53]:
for term in ["chem", "poetry", "estock", "تمام"]:
    print(f"\n### TERM: {term}")
    for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
        in_input = df_["input_text"].astype(str).str.contains(term, case=False, na=False).sum()
        in_target = df_["target_text"].astype(str).str.contains(term, case=False, na=False).sum()
        print(f"{df_name}: input={in_input}, target={in_target}")


### TERM: chem
train: input=0, target=0
valid: input=0, target=0
test: input=0, target=0

### TERM: poetry
train: input=0, target=0
valid: input=0, target=0
test: input=0, target=0

### TERM: estock
train: input=0, target=0
valid: input=0, target=0
test: input=0, target=0

### TERM: تمام
train: input=1, target=0
valid: input=1, target=0
test: input=0, target=0


In [54]:
def clean_target_text(text: str) -> str:
    text = str(text)

    # حذف كلمة تمام
    text = re.sub(r"\bتمام\b", " ", text)

    # حذف أي كلمة فيها أحرف لاتينية
    text = re.sub(r"\b[A-Za-z]+\b", " ", text)

    # تنظيف المسافات
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df_ in [train_df, valid_df, test_df]:
    df_["target_text"] = df_["target_text"].astype(str).apply(clean_target_text)

In [55]:
def token_len(x):
    return len(str(x).split())

for name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    before = len(df_)
    df_.drop(df_[df_["target_text"].astype(str).str.strip() == ""].index, inplace=True)
    df_.drop(df_[df_["target_text"].apply(token_len) < 2].index, inplace=True)
    df_.reset_index(drop=True, inplace=True)
    print(name, "before:", before, "after:", len(df_))

train before: 122 after: 122
valid before: 13 after: 13
test before: 13 after: 13


In [56]:
for df_name, df_ in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    mask = df_["target_text"].astype(str).str.contains(r"[A-Za-z]", regex=True, na=False)
    print(df_name, "remaining Latin in target:", mask.sum())

train remaining Latin in target: 0
valid remaining Latin in target: 0
test remaining Latin in target: 0
